# 01 · EDA, label mapping, preprocessing & frozen splits

**Project:** Emotional & Mental Health Monitoring — a fine-grained, trend-aware NLP framework

This notebook builds the single unified dataset that every later notebook uses.

| Step | What happens |
|---|---|
| 1 | Load **GoEmotions** (27 → 8 classes) and **Dreaddit** (stressed posts → `Stress`) |
| 2 | Clean + deduplicate, build two text views: `text_tf` (transformers / Bi-LSTM) and `text_clean` (classical TF-IDF) |
| 3 | EDA: class imbalance, length distribution, source confound |
| 4 | **Stratified 80/10/10 split, created once and frozen** (never overwritten unless `FORCE_REBUILD=True`) |

**Outputs:** `data/processed/unified.csv`, `data/processed/label_mapping.json`, `data/splits/{train,val,test}.csv`, figures in `results/figures/`.

> ⚠️ Dreaddit is not always available on the Hugging Face Hub. If the loader cell fails, download `dreaddit-train.csv` / `dreaddit-test.csv` (e.g. from Kaggle) and drop them into `data/raw/`.

In [ ]:
# Colab already has most of these; run once. Locally: pip install -r requirements.txt
!pip -q install datasets emoji nltk scikit-learn seaborn

In [ ]:
import os, sys, json, random, time, warnings, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Project root: Google Drive folder on Colab, repo root locally ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/mental-health-nlp")   # <- change if you keep the repo elsewhere
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for d in ["data/raw", "data/interim", "data/processed", "data/splits",
          "models/classical", "models/bilstm", "models/distilbert", "models/roberta",
          "results/figures", "results/reports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SEED = 42
EMOTIONS = ["Joy", "Sadness", "Anger", "Fear", "Anxiety", "Stress", "Surprise", "Neutral"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}
NEGATIVE = ["Sadness", "Anger", "Fear", "Anxiety", "Stress"]      # N in the paper (Sec. 5.6)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("ROOT =", ROOT)

## 1 · Preprocessing functions
Follows Section 5.1 of the paper. Two views of every text are produced:

* `text_tf` — URLs/HTML/handles removed, emojis → words, punctuation normalised. Case preserved. Used by Bi-LSTM and transformers.
* `text_clean` — additionally lowercased, stop-words removed (**negations kept**), lemmatised. Used by TF-IDF models only.

In [ ]:
import re, emoji, nltk
from functools import lru_cache
for _pkg in ["stopwords", "wordnet", "omw-1.4"]:
    nltk.download(_pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

URL_RE    = re.compile(r"https?://\S+|www\.\S+")
HTML_RE   = re.compile(r"<[^>]+>|&\w+;")
HANDLE_RE = re.compile(r"(?<!\w)@\w+|(?<!\w)/?u/\w+")
REPEAT_RE = re.compile(r"([!?.])\1{2,}")
WS_RE     = re.compile(r"\s+")
TOKEN_RE  = re.compile(r"[a-z]+(?:'[a-z]+)?")

# Negations are KEPT even in the classical pipeline ("not happy" must not become "happy").
_sw = set(stopwords.words("english"))
_NEG_BASE = {"no", "nor", "not", "never", "none", "nothing", "neither", "nobody", "cannot",
             "don", "didn", "doesn", "isn", "wasn", "aren", "couldn", "shouldn", "wouldn",
             "hasn", "haven", "hadn", "won", "weren", "needn", "mustn", "shan", "mightn", "ain"}
NEGATIONS  = {w for w in _sw if w in _NEG_BASE or "n't" in w} | _NEG_BASE
STOP_WORDS = _sw - NEGATIONS
_lemm = WordNetLemmatizer()

def preprocess_tf(text):
    """Light cleaning for transformers / Bi-LSTM: no lowercasing, no stop-word removal, no lemmatisation."""
    t = str(text)
    t = URL_RE.sub(" ", t); t = HTML_RE.sub(" ", t); t = HANDLE_RE.sub(" ", t)
    t = emoji.demojize(t, delimiters=(" ", " ")).replace("_", " ")      # emoji -> emotion-bearing tokens
    t = REPEAT_RE.sub(r"\1\1", t)                                       # punctuation normalisation
    return WS_RE.sub(" ", t).strip()

@lru_cache(maxsize=200_000)
def _lemma(w):
    return _lemm.lemmatize(w)

def preprocess_classical(text):
    """Full pipeline for TF-IDF models: lowercase, strip punctuation, drop stop-words (not negations), lemmatise."""
    t = preprocess_tf(text).lower().replace("\u2019", "'")
    return " ".join(_lemma(w) for w in TOKEN_RE.findall(t) if w not in STOP_WORDS)

In [ ]:
for s in ["I'm NOT happy at all!!!! 😭 check https://x.co @bob",
          "can't sleep, heart racing before the exam :( ",
          "I didn't expect that, wow 😮"]:
    print("RAW :", s)
    print("TF  :", preprocess_tf(s))
    print("CLS :", preprocess_classical(s), "\n")

## 2 · GoEmotions → 8-class taxonomy
GoEmotions has no *Anxiety* or *Stress* label. `nervousness → Anxiety`, `fear → Fear`; **Stress comes from Dreaddit**.
`confusion`, `curiosity`, `desire` have no clean home in the 8 classes and are **dropped**. Only single-label comments are kept.

The mapping below is a set of judgement calls (documented in the paper, Sec. 4.2). **Fix it now and do not tune it against test results.**

In [ ]:
from datasets import load_dataset

GOEMO_MAP = {
    # Joy
    "joy": "Joy", "amusement": "Joy", "excitement": "Joy", "gratitude": "Joy", "love": "Joy",
    "optimism": "Joy", "pride": "Joy", "relief": "Joy", "admiration": "Joy", "caring": "Joy",
    # Sadness
    "sadness": "Sadness", "grief": "Sadness", "disappointment": "Sadness", "remorse": "Sadness",
    "embarrassment": "Sadness",
    # Anger
    "anger": "Anger", "annoyance": "Anger", "disapproval": "Anger", "disgust": "Anger",
    # Fear / Anxiety
    "fear": "Fear", "nervousness": "Anxiety",
    # Surprise
    "surprise": "Surprise", "realization": "Surprise",
    # Neutral
    "neutral": "Neutral", "approval": "Neutral",
    # dropped: confusion, curiosity, desire
}

go = load_dataset("google-research-datasets/go_emotions", "simplified")
go_names = go["train"].features["labels"].feature.names
go_df = pd.concat([go[s].to_pandas() for s in go], ignore_index=True)         # we re-split ourselves
go_df = go_df[go_df["labels"].apply(len) == 1].copy()                         # single-label only
go_df["go_label"] = go_df["labels"].apply(lambda l: go_names[int(l[0])])
go_df["label"] = go_df["go_label"].map(GOEMO_MAP)
print("GoEmotions single-label comments:", len(go_df), "| dropped (unmapped):", go_df["label"].isna().sum())
go_df = go_df.dropna(subset=["label"])
go_df = go_df.assign(source="goemotions", id=lambda d: "go_" + d["id"].astype(str))[["id", "source", "text", "label"]]
go_df["label"].value_counts()

## 3 · Dreaddit → `Stress`
Only posts labelled *stressed* (`label == 1`) become `Stress`. Non-stressed posts are **not** assumed Neutral and are discarded.
Posts are long, so they are truncated to 128 words (≈ the 128-token budget in the roadmap).

*Known confound:* Dreaddit posts are longer/more personal than GoEmotions comments, so a model could learn "long text ⇒ Stress". Truncation reduces this and notebook 05 checks it explicitly.

In [ ]:
MAX_WORDS = 128
ALLOW_MISSING_DREADDIT = False     # True -> continue with 7 classes (Stress empty); only for debugging

def _normalise_dreaddit(df):
    cols = {c.lower(): c for c in df.columns}
    text_col = cols.get("text") or cols.get("post") or cols.get("sentence")
    label_col = cols.get("label")
    assert text_col and label_col, f"Unexpected Dreaddit columns: {list(df.columns)}"
    return df.rename(columns={text_col: "text", label_col: "label_raw"})[["text", "label_raw"]]

def load_dreaddit():
    # 1) local CSVs in data/raw (Kaggle format: columns include `text`, `label`)
    files = sorted(set(list((ROOT / "data/raw").glob("*readdit*.csv")) + list((ROOT / "data/raw").glob("*READDIT*.csv"))))
    if files:
        print("Using local Dreaddit files:", [f.name for f in files])
        return pd.concat([_normalise_dreaddit(pd.read_csv(f)) for f in files], ignore_index=True)
    # 2) Hugging Face mirrors (names change over time - edit this list if needed)
    for name in ["asarvazyan/dreaddit", "dreaddit"]:
        try:
            ds = load_dataset(name)
            print("Loaded Dreaddit from HF:", name)
            return pd.concat([_normalise_dreaddit(ds[s].to_pandas()) for s in ds], ignore_index=True)
        except Exception as e:
            print(f"  HF '{name}' failed: {type(e).__name__}")
    return None

dd = load_dreaddit()
if dd is None:
    msg = "Dreaddit not found. Put dreaddit-train.csv / dreaddit-test.csv into data/raw/ and re-run."
    if ALLOW_MISSING_DREADDIT:
        print("WARNING:", msg); dd_df = pd.DataFrame(columns=["id", "source", "text", "label"])
    else:
        raise FileNotFoundError(msg)
else:
    dd = dd[dd["label_raw"] == 1].copy()
    dd["text"] = dd["text"].astype(str).apply(lambda t: " ".join(t.split()[:MAX_WORDS]))
    dd_df = pd.DataFrame({"id": [f"dd_{i}" for i in range(len(dd))], "source": "dreaddit",
                          "text": dd["text"].values, "label": "Stress"})
print("Dreaddit stressed posts:", len(dd_df))

## 4 · Unify, clean, deduplicate

In [ ]:
uni = pd.concat([go_df, dd_df], ignore_index=True)
uni["text"] = uni["text"].astype(str)

# two text views
uni["text_tf"] = uni["text"].apply(preprocess_tf)
uni["text_clean"] = uni["text"].apply(preprocess_classical)

# drop empty / duplicate texts (duplicates across splits would leak)
n0 = len(uni)
uni = uni[uni["text_tf"].str.split().str.len() >= 1]
uni = uni.drop_duplicates(subset="text_tf", keep="first").reset_index(drop=True)
print(f"rows: {n0} -> {len(uni)} after cleaning/dedup")

uni["label_id"] = uni["label"].map(LABEL2ID)
uni["n_words"] = uni["text_tf"].str.split().str.len()
assert uni["label_id"].notna().all()
uni.to_csv(ROOT / "data/processed/unified.csv", index=False)
json.dump({"goemotions_to_8class": GOEMO_MAP, "dropped": ["confusion", "curiosity", "desire"],
           "dreaddit": "label==1 -> Stress; label==0 discarded; truncated to 128 words"},
          open(ROOT / "data/processed/label_mapping.json", "w"), indent=2)
uni.head()

## 5 · EDA — class imbalance & text length

In [ ]:
counts = pd.crosstab(uni["label"], uni["source"]).reindex(EMOTIONS).fillna(0).astype(int)
counts["total"] = counts.sum(axis=1)
counts["share_%"] = (100 * counts["total"] / counts["total"].sum()).round(2)
display(counts)
print("Imbalance ratio (max/min class):", round(counts["total"].max() / counts["total"].min(), 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
counts.drop(columns=["total", "share_%"]).plot(kind="bar", stacked=True, ax=axes[0], rot=30)
axes[0].set_title("Class distribution by source (before any mitigation)"); axes[0].set_ylabel("# texts")
sns.histplot(data=uni, x="n_words", hue="source", bins=40, ax=axes[1], element="step", stat="density", common_norm=False)
axes[1].set_title("Text length (words)"); axes[1].set_xlim(0, 140)
plt.tight_layout(); plt.savefig(ROOT / "results/figures/01_class_distribution_and_length.png", dpi=150); plt.show()

In [ ]:
# Sample texts per class — sanity-check the mapping by eye
for e in EMOTIONS:
    print(f"\n=== {e} ===")
    for t in uni[uni.label == e].sample(3, random_state=SEED)["text_tf"]:
        print("  -", t[:140])

## 6 · Stratified 80/10/10 split — created **once**, then frozen
Everything later (class weighting, oversampling, augmentation) is applied to the **training split only**.
Validation is used for model selection; the test set is touched only for final reporting.

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_DIR = ROOT / "data/splits"
FORCE_REBUILD = False          # DO NOT set True after you have started training models

if (SPLIT_DIR / "train.csv").exists() and not FORCE_REBUILD:
    print("Splits already exist -> NOT overwriting (they are frozen).")
    train, val, test = [pd.read_csv(SPLIT_DIR / f"{s}.csv") for s in ("train", "val", "test")]
else:
    train, tmp = train_test_split(uni, test_size=0.20, stratify=uni["label_id"], random_state=SEED)
    val, test = train_test_split(tmp, test_size=0.50, stratify=tmp["label_id"], random_state=SEED)
    for name, d in [("train", train), ("val", val), ("test", test)]:
        d.reset_index(drop=True).to_csv(SPLIT_DIR / f"{name}.csv", index=False)
    print("Splits written to", SPLIT_DIR)

dist = pd.DataFrame({n: d["label"].value_counts(normalize=True).reindex(EMOTIONS) for n, d in
                     [("train", train), ("val", val), ("test", test)]}).round(4)
print({n: len(d) for n, d in [("train", train), ("val", val), ("test", test)]})
display(dist)    # proportions must match across splits
assert not (set(train.id) & set(test.id)) and not (set(train.id) & set(val.id)) and not (set(val.id) & set(test.id))

In [ ]:
# Class weights  w_c = N / (|E| * n_c)   (paper Sec. 5.4) — computed from TRAIN only
N = len(train); n_c = train["label"].value_counts().reindex(EMOTIONS)
class_weights = (N / (len(EMOTIONS) * n_c)).round(4)
json.dump(class_weights.to_dict(), open(ROOT / "data/processed/class_weights.json", "w"), indent=2)
display(pd.DataFrame({"train_count": n_c, "weight": class_weights}))

### ✅ Done
Next: **`02_classical_models.ipynb`**. Remember to state the Dreaddit-confound and mapping judgement calls in the report's *Limitations* section.